# E15 — 일반 replay · seed 42
E10 유지: DDQN / 84×84 / stack 4 / skip 4 / 300,000 원본 프레임. T4 GPU 선택 후 순서대로 실행.
검증 500–504로 best 선택, 공개 비교 1000–1009. 학습 seed는 초기화·탐색·학습 트랙을 함께 변경합니다. E14는 CPU였으므로 GPU 차이도 기록합니다.

## 1. Drive와 코드 준비
최초 실행만 GitHub에서 복사. 기존 Drive 코드와 결과는 자동 갱신하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, shutil, tempfile
ROOT=Path('/content/drive/MyDrive/261103')
RESULTS=ROOT/'261103_results'
if not (ROOT/'train.py').exists():
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(['git','clone','--depth','1','--branch','261103','https://github.com/MilkySon/RL_carracing_competition.git',tmp+'/repo'],check=True)
        shutil.copytree(Path(tmp)/'repo'/'261103',ROOT,dirs_exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
os.environ['SDL_VIDEODRIVER']='dummy'
os.environ['SDL_AUDIODRIVER']='dummy'
print(ROOT,RESULTS)


## 2. 설치

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','swig'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import torch
print('GPU:',torch.cuda.is_available())
def run(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a') as f:
        p=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='');f.write(line);f.flush()
        rc=p.wait()
    if rc:raise RuntimeError(f'Failed ({rc}): {log}')


## 3. 독립 학습
중단된 기록은 새 RUN_NAME으로 처음부터 실행. 원본 resume은 replay/RNG 미복원이므로 자동 사용하지 않습니다. replay는 GPU와 별개로 호스트 RAM을 많이 사용할 수 있습니다.

In [ ]:
RUN_NAME='E15'
RUN_DIR=RESULTS/RUN_NAME
ALLOW_CPU=False
if not torch.cuda.is_available() and not ALLOW_CPU:raise RuntimeError('GPU를 선택하거나 ALLOW_CPU=True로 설정하세요.')
RUN_DIR.mkdir(parents=True,exist_ok=True)
if (RUN_DIR/'final.pt').exists():print('완료된 학습 사용:',RUN_DIR)
elif any((RUN_DIR/n).exists() for n in ['train.csv','eval.csv','latest.pt','best.pt']):
    raise RuntimeError('중단된 결과가 있습니다. 새 RUN_NAME을 사용하세요.')
else:
    import hashlib,platform
    manifest={'agent':'ddqn_e10','seed':42,'budget':300000,'python':platform.python_version(),
        'torch':torch.__version__,'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'source_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in ROOT.rglob('*.py') if '261103_results' not in p.parts}}
    (RUN_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
    run([sys.executable,'train.py','--agent','ddqn_e10','--seed',42,'--budget',300000,'--eval-every',50000,'--run-dir',RUN_DIR],RUN_DIR/'training_console.log')
assert (RUN_DIR/'best.pt').exists()


## 4. 학습 곡선

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
tr=pd.read_csv(RUN_DIR/'train.csv');ev=pd.read_csv(RUN_DIR/'eval.csv')
plt.plot(tr.frames,tr['return'].rolling(10,min_periods=1).mean(),label='train 10ep')
plt.plot(ev.frames,ev.score,'o-',label='validation 500-504')
plt.xlabel('raw frames');plt.ylabel('return');plt.legend();plt.show()
display(ev)


## 5. 평가·영상
best/final을 비교해 후반 성능 하락을 진단합니다. 2000–2029는 추가 개발 평가이며 최종 시험과 분리하세요.

In [ ]:
for d in ['reports','videos','onnx']:(RUN_DIR/d).mkdir(exist_ok=True)
for ckpt in ['best','final']:
    for seeds in ['1000-1009','2000-2029']:
        run([sys.executable,'evaluate.py','--ckpt',RUN_DIR/f'{ckpt}.pt','--seeds',seeds,'--out',RUN_DIR/'reports'/f'{ckpt}_{seeds}.json'],RUN_DIR/'evaluation_console.log')
run([sys.executable,'evaluate.py','--ckpt',RUN_DIR/'best.pt','--seeds','1000-1002','--video',RUN_DIR/'videos'],RUN_DIR/'video_console.log')
from IPython.display import Video,display
videos=sorted((RUN_DIR/'videos').glob('*.mp4'))
if videos:display(Video(str(videos[0]),embed=True,width=400))
rows=[]
for p in sorted((RUN_DIR/'reports').glob('best_*.json')):
    obj=json.loads(p.read_text());v=sorted(obj['returns']);k=max(1,len(v)//5)
    rows.append(dict(report=p.stem,mean=obj['mean'],std=obj['std'],below700=sum(x<700 for x in v),bottom20_mean=sum(v[:k])/k))
summary=pd.DataFrame(rows);display(summary);summary.to_csv(RUN_DIR/'reports'/'summary.csv',index=False)


## 6. ONNX 제출기 검증

In [ ]:
TEAM_NAME='MyTeam'
model=RUN_DIR/'onnx'/f'{TEAM_NAME}_E15.onnx'
run([sys.executable,'export.py','--ckpt',RUN_DIR/'best.pt','--out',model],RUN_DIR/'export_console.log')
run([sys.executable,'check_submission.py','--onnx',model,'--seeds','1000-1009','--json',RUN_DIR/'reports'/'onnx_1000-1009.json'],RUN_DIR/'submission_console.log')
print('다음: E16_colab_quickstart.ipynb')
